# 🐍 Python od podstaw — Moduł 9: Testowanie kodu

### Skąd wiesz, że Twoja funkcja naprawdę działa?

Do tej pory sprawdzałeś/aś, czy kod działa, patrząc na wynik `print()`. To działa przy
jednej funkcji w notebooku, ale nie skaluje się — im więcej kodu, tym trudniej ręcznie
sprawdzić wszystko po każdej zmianie. Testy to sposób na automatyczne, powtarzalne
sprawdzanie „czy to wciąż działa”.

## Spis treści

1. [Po co testować kod](#sec1)
2. [`assert` — najprostszy test](#sec2)
3. [Moduł `unittest` — podstawy](#sec3)
4. [Najważniejsze metody `assert*`](#sec4)
5. [Testowanie wyjątków](#sec5)
6. [Organizacja testów i `pytest`](#sec6)
7. [Co właściwie testować — przypadki brzegowe](#sec7)
8. [Ciekawostka: piramida testów](#sec8)
9. [Podsumowanie modułu](#sec9)
10. [Ćwiczenia](#sec10)

---

<a id="sec1"></a>
## 1. Po co testować kod

Wyobraź sobie funkcję `oblicz_bmi()` z modułu 4, używaną w większym programie. Kolejny
programista (albo Ty za pół roku) poprawia coś w tej funkcji i przypadkiem psuje wzór.
Bez testów zauważysz to dopiero, gdy coś się wywali na produkcji — albo wcale. Z
testami: uruchamiasz jedno polecenie, i w sekundę wiesz, że coś jest nie tak, zanim
ktokolwiek inny to zobaczy.

<a id="sec2"></a>
## 2. `assert` — najprostszy test

Instrukcja `assert wyrazenie` nic nie robi, jeśli `wyrazenie` jest prawdziwe (`True`),
a wywołuje błąd `AssertionError`, jeśli jest fałszywe. To fundament, na którym stoją
bardziej rozbudowane narzędzia do testowania.

In [ ]:
def dodaj(a, b):
    return a + b

assert dodaj(2, 3) == 5          # przechodzi bez żadnego komunikatu
assert dodaj(-1, 1) == 0         # też przechodzi

# To wywoła AssertionError, bo funkcja jest (celowo) błędna:
def dodaj_z_bledem(a, b):
    return a + b + 1   # błąd: dodatkowe +1

try:
    assert dodaj_z_bledem(2, 3) == 5
except AssertionError:
    print("Test nie przeszedł - funkcja zwróciła zły wynik!")

<a id="sec3"></a>
## 3. Moduł `unittest` — podstawy

`unittest` to wbudowany moduł do pisania testów w bardziej zorganizowany sposób.
Testy grupuje się w klasie dziedziczącej po `unittest.TestCase`, a każda metoda
zaczynająca się od `test_` to osobny, niezależny test.

In [ ]:
%%writefile kalkulator.py
def dodaj(a, b):
    return a + b

def odejmij(a, b):
    return a - b

In [ ]:
%%writefile test_kalkulator.py
import unittest
from kalkulator import dodaj, odejmij

class TestKalkulatora(unittest.TestCase):
    def test_dodawanie(self):
        self.assertEqual(dodaj(2, 3), 5)

    def test_dodawanie_liczb_ujemnych(self):
        self.assertEqual(dodaj(-1, -1), -2)

    def test_odejmowanie(self):
        self.assertEqual(odejmij(5, 3), 2)

if __name__ == "__main__":
    unittest.main()

In [ ]:
# Uruchomienie wszystkich testów z pliku (w terminalu wygląda tak samo):
!python -m unittest test_kalkulator.py -v

> 💡 **Skąd nazwy `test_...`**
>
> `unittest` (i `pytest`) automatycznie rozpoznają metody i pliki zaczynające się od `test_` jako testy do uruchomienia - to konwencja, nie przypadek. Dzięki temu można trzymać kod produkcyjny i testy w osobnych plikach, a narzędzie samo znajdzie, co uruchomić.

<a id="sec4"></a>
## 4. Najważniejsze metody `assert*`

`unittest.TestCase` ma całą rodzinę metod `assert*`, czytelniejszych niż goły
`assert` - m.in. dlatego, że przy niepowodzeniu pokazują dokładnie, jaka wartość była
oczekiwana, a jaka faktyczna.

| Metoda | Sprawdza |
|---|---|
| `assertEqual(a, b)` | czy `a == b` |
| `assertNotEqual(a, b)` | czy `a != b` |
| `assertTrue(x)` | czy `x` jest prawdziwe |
| `assertFalse(x)` | czy `x` jest fałszywe |
| `assertIsNone(x)` | czy `x` to `None` |
| `assertIn(a, b)` | czy `a` jest w `b` (liście, stringu...) |
| `assertRaises(Blad, ...)` | czy wywołanie zgłasza dany wyjątek |

<a id="sec5"></a>
## 5. Testowanie wyjątków

W module 5 nauczyłeś/aś się zgłaszać błędy przez `raise`. Warto też przetestować, że
funkcja **rzeczywiście** zgłasza wyjątek, gdy powinna - `assertRaises` sprawdza
dokładnie to, najczęściej jako menedżer kontekstu (`with`).

In [ ]:
%%writefile konto.py
class KontoBankowe:
    def __init__(self):
        self._saldo = 0

    def wplac(self, kwota):
        self._saldo += kwota

    def wyplac(self, kwota):
        if kwota > self._saldo:
            raise ValueError("Brak wystarczających środków")
        self._saldo -= kwota

In [ ]:
%%writefile test_konto.py
import unittest
from konto import KontoBankowe

class TestKontaBankowego(unittest.TestCase):
    def test_wplata_zwieksza_saldo(self):
        konto = KontoBankowe()
        konto.wplac(100)
        self.assertEqual(konto._saldo, 100)

    def test_wyplata_ponad_saldo_zglasza_blad(self):
        konto = KontoBankowe()
        konto.wplac(50)
        with self.assertRaises(ValueError):
            konto.wyplac(100)

if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest test_konto.py -v

<a id="sec6"></a>
## 6. Organizacja testów i `pytest`

W realnych projektach testy trzyma się zwykle w osobnym folderze `tests/`, po jednym
pliku `test_*.py` na każdy moduł produkcyjny. `pytest` to popularny pakiet zewnętrzny
(`pip install pytest` - pamiętasz moduł 7?), który robi to samo co `unittest`, ale
prostszą składnią - zwykłe `assert`, bez klas i dziedziczenia.

```python
# To samo co przykład z sekcji 3, ale w stylu pytest (plik test_kalkulator_pytest.py):
from kalkulator import dodaj, odejmij

def test_dodawanie():
    assert dodaj(2, 3) == 5

def test_odejmowanie():
    assert odejmij(5, 3) == 2
```

Uruchamia się to poleceniem `pytest` w terminalu (po `pip install pytest`) - narzędzie
samo znajdzie wszystkie pliki `test_*.py` i funkcje `test_*` w bieżącym folderze.

<a id="sec7"></a>
## 7. Co właściwie testować — przypadki brzegowe

Dobry test nie sprawdza tylko „typowego” przypadku. Warto pomyśleć o **przypadkach
brzegowych**: pustej liście, liczbie 0 albo ujemnej, bardzo dużej wartości, stringu
pustym albo z samymi spacjami. To właśnie tam najczęściej ukrywają się błędy.

In [ ]:
def srednia(liczby):
    return sum(liczby) / len(liczby)

print(srednia([1, 2, 3]))   # typowy przypadek - działa

# A co z pustą listą?
try:
    print(srednia([]))
except ZeroDivisionError:
    print("Pusta lista powoduje dzielenie przez zero - to warto przetestować i obsłużyć!")

> ⚠️ **Test, który nigdy nie może nie przejść, jest bezwartościowy**
>
> Pisanie `assertEqual(2 + 2, 4)` w teście dla Twojej funkcji nic nie sprawdza - test musi faktycznie wywoływać Twój kod i sprawdzać jego wynik, najlepiej z wieloma różnymi, sensownie dobranymi danymi wejściowymi.

<a id="sec8"></a>
## 8. Ciekawostka: piramida testów

W większych projektach testy dzieli się zwykle na warstwy.

> 💡 **Ciekawostka**
>
> «Piramida testów» to popularny model: u podstawy dużo szybkich **testów jednostkowych** (jak te w tym module - pojedyncza funkcja, izolowana), wyżej mniej **testów integracyjnych** (sprawdzających, czy kilka elementów współpracuje poprawnie), a na szczycie nieliczne, wolniejsze **testy end-to-end** (sprawdzające cały program z perspektywy użytkownika). Im niżej w piramidzie, tym test powinien być szybszy i tańszy w utrzymaniu.

<a id="sec9"></a>
## 9. Podsumowanie modułu

Po tym module powinno być jasne:

- jak działa `assert` i czym różni się od `assertEqual` z `unittest`,
- jak zorganizować testy w klasie `unittest.TestCase`,
- najważniejsze metody `assert*` (`assertEqual`, `assertTrue`, `assertRaises`...),
- jak przetestować, że funkcja poprawnie zgłasza wyjątek,
- czym różni się `unittest` od `pytest`,
- dlaczego warto testować przypadki brzegowe, nie tylko typowy scenariusz.

Ostatni moduł tej serii: **praca z API i wprowadzenie do pandas** — jak pobierać dane z
internetu i analizować je, łącząc wszystko, czego się do tej pory nauczyłeś/aś.

<a id="sec10"></a>
## 10. Ćwiczenia

Kilka zadań wraca do funkcji z wcześniejszych modułów - tym razem po to, żeby napisać
dla nich testy, a nie tylko je wywołać.

> 📝 **Ćwiczenie 1: Testy dla `czy_parzysta`**
>
> Stwórz plik `parzystosc.py` z funkcją `czy_parzysta(n)` z modułu 4. Napisz do niej `unittest.TestCase` z co najmniej dwoma testami: liczba parzysta i liczba nieparzysta. Uruchom testy przez `!python -m unittest`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile parzystosc.py
# def czy_parzysta(n):
#     return n % 2 == 0

# %%writefile test_parzystosc.py
# import unittest
# from parzystosc import czy_parzysta
#
# class TestParzystosci(unittest.TestCase):
#     def test_liczba_parzysta(self):
#         self.assertTrue(czy_parzysta(4))
#
#     def test_liczba_nieparzysta(self):
#         self.assertFalse(czy_parzysta(7))
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_parzystosc.py -v
```
</details>

> 📝 **Ćwiczenie 2: Testy dla kalkulatora BMI**
>
> Stwórz plik `bmi.py` z funkcją `oblicz_bmi(waga, wzrost)` z modułu 4. Napisz test sprawdzający, że dla `waga=70, wzrost=1.75` wynik jest w przybliżeniu równy `22.86` - użyj `assertAlmostEqual(wynik, 22.86, places=2)`, bo porównywanie liczb zmiennoprzecinkowych przez `==` bywa zawodne.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile bmi.py
# def oblicz_bmi(waga, wzrost):
#     return waga / wzrost ** 2

# %%writefile test_bmi.py
# import unittest
# from bmi import oblicz_bmi
#
# class TestBmi(unittest.TestCase):
#     def test_typowy_przypadek(self):
#         wynik = oblicz_bmi(70, 1.75)
#         self.assertAlmostEqual(wynik, 22.86, places=2)
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_bmi.py -v
```

Podpowiedź: `assertAlmostEqual` porównuje liczby z tolerancją, co jest kluczowe przy liczbach zmiennoprzecinkowych - `0.1 + 0.2 == 0.3` w Pythonie daje `False` z powodu sposobu przechowywania ułamków w pamięci komputera.
</details>

> 📝 **Ćwiczenie 3: Testowanie przypadku brzegowego - pusta lista**
>
> Napisz funkcję `srednia_bezpieczna(liczby)`, która zwraca `None` dla pustej listy zamiast wywalać się `ZeroDivisionError` (podpowiedź: `if not liczby: return None`). Napisz DWA testy: jeden dla normalnej listy, drugi sprawdzający zachowanie dla listy pustej.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile srednia.py
# def srednia_bezpieczna(liczby):
#     if not liczby:
#         return None
#     return sum(liczby) / len(liczby)

# %%writefile test_srednia.py
# import unittest
# from srednia import srednia_bezpieczna
#
# class TestSredniej(unittest.TestCase):
#     def test_typowa_lista(self):
#         self.assertEqual(srednia_bezpieczna([1, 2, 3]), 2)
#
#     def test_pusta_lista(self):
#         self.assertIsNone(srednia_bezpieczna([]))
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_srednia.py -v
```
</details>

> 📝 **Ćwiczenie 4: Testowanie zgłaszanego wyjątku**
>
> Mając klasę `KontoBankowe` z sekcji 5, napisz dodatkowy test sprawdzający, że próba wpłaty **ujemnej** kwoty zgłasza `ValueError` - w tym celu najpierw dodaj taką walidację do metody `wplac()`, a potem napisz do niej test z `assertRaises`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# W konto.py, metoda wplac() powinna zacząć się od:
# def wplac(self, kwota):
#     if kwota < 0:
#         raise ValueError("Kwota wpłaty nie może być ujemna")
#     self._saldo += kwota

# W teście:
# def test_wplata_ujemnej_kwoty_zglasza_blad(self):
#     konto = KontoBankowe()
#     with self.assertRaises(ValueError):
#         konto.wplac(-50)
```
</details>

> 📝 **Ćwiczenie 5: Testy dla walidacji hasła**
>
> Mając funkcję `ustaw_haslo(haslo)` z modułu 5 (zgłasza `ValueError`, jeśli hasło ma mniej niż 8 znaków), napisz dwa testy: jeden dla poprawnego hasła (sprawdza zwróconą wartość), drugi dla za krótkiego hasła (sprawdza, że zgłasza `ValueError`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile haslo.py
# def ustaw_haslo(haslo):
#     if len(haslo) < 8:
#         raise ValueError("Hasło musi mieć co najmniej 8 znaków")
#     return "Hasło ustawione"

# %%writefile test_haslo.py
# import unittest
# from haslo import ustaw_haslo
#
# class TestHasla(unittest.TestCase):
#     def test_poprawne_haslo(self):
#         self.assertEqual(ustaw_haslo("bezpieczne123"), "Hasło ustawione")
#
#     def test_za_krotkie_haslo(self):
#         with self.assertRaises(ValueError):
#             ustaw_haslo("abc")
#
# if __name__ == "__main__":
#     unittest.main()
```
</details>

> 📝 **Ćwiczenie 6: Testy dla `is_palindrome`**
>
> Mając funkcję `czy_palindrom(s)` z modułu 7, napisz co najmniej trzy testy: palindrom prosty (np. `"kajak"`), NIE-palindrom (np. `"python"`), i przypadek z wielkimi literami (np. `"Kajak"` też powinno być uznane za palindrom).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile tekst_util.py
# def czy_palindrom(s):
#     s = s.lower()
#     return s == s[::-1]

# %%writefile test_tekst_util.py
# import unittest
# from tekst_util import czy_palindrom
#
# class TestPalindromu(unittest.TestCase):
#     def test_prosty_palindrom(self):
#         self.assertTrue(czy_palindrom("kajak"))
#
#     def test_nie_palindrom(self):
#         self.assertFalse(czy_palindrom("python"))
#
#     def test_wielkie_litery(self):
#         self.assertTrue(czy_palindrom("Kajak"))
#
# if __name__ == "__main__":
#     unittest.main()
```
</details>

> 📝 **Ćwiczenie 7: Test parametryzowany ręcznie**
>
> Mając funkcję `czy_pierwsza(n)` z modułu 7, napisz JEDEN test, który w pętli sprawdza listę znanych liczb pierwszych `[2, 3, 5, 7, 11, 13]` (wszystkie powinny dać `True`) oraz listę znanych liczb złożonych `[4, 6, 8, 9, 10]` (wszystkie powinny dać `False`) - bez pisania osobnej metody `test_` dla każdej liczby.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile liczby_util.py
# def czy_pierwsza(n):
#     if n < 2:
#         return False
#     for dzielnik in range(2, int(n ** 0.5) + 1):
#         if n % dzielnik == 0:
#             return False
#     return True

# %%writefile test_liczby_util.py
# import unittest
# from liczby_util import czy_pierwsza
#
# class TestLiczbPierwszych(unittest.TestCase):
#     def test_liczby_pierwsze(self):
#         for liczba in [2, 3, 5, 7, 11, 13]:
#             self.assertTrue(czy_pierwsza(liczba), f"{liczba} powinna być pierwsza")
#
#     def test_liczby_zlozone(self):
#         for liczba in [4, 6, 8, 9, 10]:
#             self.assertFalse(czy_pierwsza(liczba), f"{liczba} nie powinna być pierwsza")
#
# if __name__ == "__main__":
#     unittest.main()
```

Podpowiedź: drugi argument `assertTrue`/`assertFalse` to opcjonalny komunikat wyświetlany, gdy test nie przejdzie - bardzo pomocny właśnie w pętli, żeby wiedzieć DLA KTÓREJ liczby test faktycznie zawiódł.
</details>

> 🔥 **Ćwiczenie 8 (wyzwanie): TDD - najpierw test, potem kod**
>
> Tym razem odwróć kolejność: napisz NAJPIERW testy dla funkcji `policz_wystapienia(tekst, slowo)` (ma zwracać, ile razy dane słowo występuje w tekście, bez rozróżniania wielkości liter), obejmujące: typowy przypadek, słowo nieobecne w tekście (wynik 0), i pusty tekst. Dopiero PO napisaniu testów zaimplementuj funkcję tak, żeby wszystkie przeszły.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# Krok 1 - testy PRZED implementacją (będą czerwone, dopóki nie ma jeszcze funkcji):
# %%writefile test_liczenie_slow.py
# import unittest
# from liczenie_slow import policz_wystapienia
#
# class TestLiczeniaSlow(unittest.TestCase):
#     def test_typowy_przypadek(self):
#         tekst = "python jest super, uczę się pythona, kocham Python"
#         self.assertEqual(policz_wystapienia(tekst, "python"), 3)
#
#     def test_slowo_nieobecne(self):
#         self.assertEqual(policz_wystapienia("jakiś tekst", "brak"), 0)
#
#     def test_pusty_tekst(self):
#         self.assertEqual(policz_wystapienia("", "cokolwiek"), 0)
#
# if __name__ == "__main__":
#     unittest.main()

# Krok 2 - dopiero teraz implementacja, napisana tak, by testy przeszły:
# %%writefile liczenie_slow.py
# def policz_wystapienia(tekst, slowo):
#     slowa = tekst.lower().split()
#     return slowa.count(slowo.lower())

# !python -m unittest test_liczenie_slow.py -v
```

To jest esencja TDD (Test-Driven Development) - testy definiują, czego oczekujesz od kodu, ZANIM ten kod istnieje. Wymusza to myślenie o przypadkach brzegowych na starcie, a nie jako refleksję po fakcie.
</details>

---

### Co dalej?

Jeśli ćwiczenie TDD (testy przed kodem) poszło w miarę gładko, masz solidne
podstawy do pisania kodu, któremu można zaufać. Ostatni, dziesiąty moduł tej serii
połączy wszystko: pracę z zewnętrznym API (`requests`) i wprowadzenie do analizy danych
(`pandas`).